# SELFIES Feature Engineering

Bu notebook'ta node ve linker SMILES ifadeleri ayrı ayrı
SELFIES gösterimine dönüştürülmekte ve token dizileri oluşturulmaktadır.

In [1]:
from pathlib import Path
from functools import lru_cache
from collections import Counter
from importlib.metadata import version
import ast
import json

import pandas as pd
import selfies as sf


PROJECT_ROOT = Path.cwd().parent

INPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "forward_model_main.csv"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

model_df = pd.read_csv(
    INPUT_PATH,
    low_memory=False,
)

print("SELFIES sürümü:", version("selfies"))
print("Veri seti boyutu:", model_df.shape)

display(model_df.head())

SELFIES sürümü: 2.2.0
Veri seti boyutu: (17544, 9)


,qmof_id,smiles_nodes,smiles_linkers,point_group,topology,density,pld,lcd,topology_missing
0,qmof-8a95c27,"['O', '[Ba]', '[Cu]']",['[O-]C=O'],-1,__MISSING_TOPOLOGY__,2.763246,0.68822,1.35480,1
1,qmof-830ed1c,['[Co]'],['[O-]C(=O)c1ccncc1'],m,rtl,1.557644,2.36128,4.21176,0
2,qmof-5bd4a24,['[Co]'],['[O-]C(=O)c1ccncc1'],2/m,rtl,1.616139,2.14542,3.27957,0
3,qmof-644aab4,['[Zn][Zn]'],"['[O-]C(=O)c1cccc(c1)c1nccs1', 'n1ccc(cc1)c1cc...",-1,__MISSING_TOPOLOGY__,1.596537,1.33452,2.03948,1
4,qmof-eaa4957,"['[OH2][Ag][Ag][Ag][Ag][OH2]', '[OH2][Ag][Ag][...","['[O-]C(=O)C1=NN=C([CH]1)C(=O)[O-]', '[O-]C(=O...",-1,__MISSING_TOPOLOGY__,3.421198,0.78190,2.42132,1


In [2]:
def parse_smiles_list(value):
    """
    CSV hücresindeki SMILES bilgisini güvenli şekilde
    Python listesine dönüştürür.
    """

    if isinstance(value, (list, tuple)):
        parsed_items = list(value)

    elif pd.isna(value):
        return []

    elif isinstance(value, str):
        text = value.strip()

        if not text:
            return []

        try:
            parsed_value = ast.literal_eval(text)
        except (ValueError, SyntaxError):
            # Hücre liste değil, tek bir SMILES olabilir.
            parsed_value = text

        if isinstance(parsed_value, (list, tuple)):
            parsed_items = list(parsed_value)
        else:
            parsed_items = [parsed_value]

    else:
        parsed_items = [value]

    cleaned_items = []

    for item in parsed_items:
        if item is None:
            continue

        item_text = str(item).strip()

        if item_text.lower() in {"", "none", "nan"}:
            continue

        cleaned_items.append(item_text)

    return cleaned_items

In [3]:
example_values = [
    "['[Co]']",
    "['[Co]', '[Zn]']",
    "['[O-]C(=O)c1ccncc1']",
    "[Cu]",
    None,
]

for value in example_values:
    print("Girdi :", value)
    print("Çıktı:", parse_smiles_list(value))
    print("-" * 50)

Girdi : ['[Co]']
Çıktı: ['[Co]']
--------------------------------------------------
Girdi : ['[Co]', '[Zn]']
Çıktı: ['[Co]', '[Zn]']
--------------------------------------------------
Girdi : ['[O-]C(=O)c1ccncc1']
Çıktı: ['[O-]C(=O)c1ccncc1']
--------------------------------------------------
Girdi : [Cu]
Çıktı: ['[Cu]']
--------------------------------------------------
Girdi : None
Çıktı: []
--------------------------------------------------


In [4]:
MOLECULE_SEPARATOR_TOKEN = "<MOL_SEP>"


def encode_smiles_collection(value):
    """
    Bir hücre içerisindeki bir veya daha fazla SMILES ifadesini
    ayrı ayrı SELFIES'e çevirir.
    """

    smiles_list = parse_smiles_list(value)

    selfies_list = []
    combined_tokens = []
    failed_smiles = []

    strict_count = 0
    relaxed_count = 0

    for smiles in smiles_list:
        result = encode_single_smiles(smiles)

        if result["status"] == "failed":
            failed_smiles.append({
                "smiles": smiles,
                "error": result["error"],
            })
            continue

        if result["status"] == "strict":
            strict_count += 1
        elif result["status"] == "relaxed":
            relaxed_count += 1

        if combined_tokens:
            combined_tokens.append(
                MOLECULE_SEPARATOR_TOKEN
            )

        selfies_list.append(result["selfies"])
        combined_tokens.extend(result["tokens"])

    return {
        "smiles_list": smiles_list,
        "selfies_list": selfies_list,
        "tokens": combined_tokens,
        "strict_count": strict_count,
        "relaxed_count": relaxed_count,
        "failed_smiles": failed_smiles,
    }

In [5]:
from functools import lru_cache
import selfies as sf


@lru_cache(maxsize=None)
def encode_single_smiles(smiles):
    """
    Tek bir SMILES ifadesini SELFIES ve token dizisine çevirir.
    """

    try:
        selfies_string = sf.encoder(smiles)

        return {
            "smiles": smiles,
            "selfies": selfies_string,
            "tokens": tuple(sf.split_selfies(selfies_string)),
            "status": "strict",
            "error": None,
        }

    except sf.EncoderError as strict_error:
        try:
            selfies_string = sf.encoder(
                smiles,
                strict=False,
            )

            return {
                "smiles": smiles,
                "selfies": selfies_string,
                "tokens": tuple(sf.split_selfies(selfies_string)),
                "status": "relaxed",
                "error": str(strict_error),
            }

        except Exception as relaxed_error:
            return {
                "smiles": smiles,
                "selfies": None,
                "tokens": tuple(),
                "status": "failed",
                "error": str(relaxed_error),
            }

    except Exception as unexpected_error:
        return {
            "smiles": smiles,
            "selfies": None,
            "tokens": tuple(),
            "status": "failed",
            "error": str(unexpected_error),
        }

In [6]:
node_encoding_results = (
    model_df["smiles_nodes"]
    .apply(encode_smiles_collection)
)

model_df["node_smiles_list"] = node_encoding_results.apply(
    lambda result: result["smiles_list"]
)

model_df["node_selfies_list"] = node_encoding_results.apply(
    lambda result: result["selfies_list"]
)

model_df["node_tokens"] = node_encoding_results.apply(
    lambda result: result["tokens"]
)

model_df["node_relaxed_count"] = node_encoding_results.apply(
    lambda result: result["relaxed_count"]
)

model_df["node_failed_count"] = node_encoding_results.apply(
    lambda result: len(result["failed_smiles"])
)

print("Node dönüşümü tamamlandı.")

Node dönüşümü tamamlandı.


In [7]:
linker_encoding_results = (
    model_df["smiles_linkers"]
    .apply(encode_smiles_collection)
)

model_df["linker_smiles_list"] = linker_encoding_results.apply(
    lambda result: result["smiles_list"]
)

model_df["linker_selfies_list"] = linker_encoding_results.apply(
    lambda result: result["selfies_list"]
)

model_df["linker_tokens"] = linker_encoding_results.apply(
    lambda result: result["tokens"]
)

model_df["linker_relaxed_count"] = linker_encoding_results.apply(
    lambda result: result["relaxed_count"]
)

model_df["linker_failed_count"] = linker_encoding_results.apply(
    lambda result: len(result["failed_smiles"])
)

print("Linker dönüşümü tamamlandı.")

Linker dönüşümü tamamlandı.


In [8]:
model_df["node_token_length"] = (
    model_df["node_tokens"].str.len()
)

model_df["linker_token_length"] = (
    model_df["linker_tokens"].str.len()
)

model_df["selfies_encoding_ok"] = (
    (model_df["node_failed_count"] == 0)
    & (model_df["linker_failed_count"] == 0)
    & (model_df["node_token_length"] > 0)
    & (model_df["linker_token_length"] > 0)
)

print(
    "Başarılı kayıt sayısı:",
    model_df["selfies_encoding_ok"].sum(),
)

print(
    "Sorunlu kayıt sayısı:",
    (~model_df["selfies_encoding_ok"]).sum(),
)

print(
    "Başarı oranı:",
    round(
        model_df["selfies_encoding_ok"].mean() * 100,
        2,
    ),
    "%",
)

print(
    "Relaxed dönüşüm kullanılan node sayısı:",
    model_df["node_relaxed_count"].sum(),
)

print(
    "Relaxed dönüşüm kullanılan linker sayısı:",
    model_df["linker_relaxed_count"].sum(),
)

Başarılı kayıt sayısı: 17348
Sorunlu kayıt sayısı: 196
Başarı oranı: 98.88 %
Relaxed dönüşüm kullanılan node sayısı: 5751
Relaxed dönüşüm kullanılan linker sayısı: 1771


In [9]:
failure_records = []

for row_index, result in node_encoding_results.items():
    qmof_id = model_df.loc[row_index, "qmof_id"]

    for failed_item in result["failed_smiles"]:
        failure_records.append({
            "qmof_id": qmof_id,
            "input_type": "node",
            "smiles": failed_item["smiles"],
            "error": failed_item["error"],
        })

for row_index, result in linker_encoding_results.items():
    qmof_id = model_df.loc[row_index, "qmof_id"]

    for failed_item in result["failed_smiles"]:
        failure_records.append({
            "qmof_id": qmof_id,
            "input_type": "linker",
            "smiles": failed_item["smiles"],
            "error": failed_item["error"],
        })

failure_df = pd.DataFrame(failure_records)

print("Toplam başarısız SMILES:", len(failure_df))

display(failure_df.head(20))

Toplam başarısız SMILES: 201


,qmof_id,input_type,smiles,error
0,qmof-72352f8,linker,C(Cn1[n-]ccc1)CCn1[n-]ccc1,kekulization failed\n\tSMILES: C(Cn1[n-]ccc1)C...
1,qmof-4063c99,linker,C(Cn1[n-]ccc1)CCn1[n-]ccc1,kekulization failed\n\tSMILES: C(Cn1[n-]ccc1)C...
2,qmof-292e1e6,linker,Cc1cc([n-]n1Cc1ccc(cc1)Cn1[n-]c(cc1C)C)C,kekulization failed\n\tSMILES: Cc1cc([n-]n1Cc1...
3,qmof-d54431e,linker,Cc1cc([n-]n1Cc1ccc(cc1)Cn1[n-]c(cc1C)C)C,kekulization failed\n\tSMILES: Cc1cc([n-]n1Cc1...
4,qmof-9c5c154,linker,c1ccn([n-]1)Cc1ccc(cc1)Cn1[n-]ccc1,kekulization failed\n\tSMILES: c1ccn([n-]1)Cc1...
5,qmof-75d4f07,linker,Cc1cc([n-]n1Cc1ccc(cc1)Cn1[n-]c(cc1C)C)C,kekulization failed\n\tSMILES: Cc1cc([n-]n1Cc1...
6,qmof-c80c803,linker,Cc1cc([n-]n1Cc1ccc(cc1)Cn1[n-]c(cc1C)C)C,kekulization failed\n\tSMILES: Cc1cc([n-]n1Cc1...
7,qmof-6e151eb,linker,Cc1cc([n-]n1Cc1ccc(cc1)Cn1[n-]c(cc1C)C)C,kekulization failed\n\tSMILES: Cc1cc([n-]n1Cc1...
8,qmof-9f8456f,linker,Cc1[nH][n-]c(c1Cc1c(C)[n-][nH]c1C)C,kekulization failed\n\tSMILES: Cc1[nH][n-]c(c1...
9,qmof-7003032,linker,[n-]1[nH]cc(c1)Cc1c[nH][n-]c1,kekulization failed\n\tSMILES: [n-]1[nH]cc(c1)...


In [10]:
length_statistics = model_df[
    [
        "node_token_length",
        "linker_token_length",
    ]
].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
).T

display(length_statistics)

,count,mean,std,min,50%,75%,90%,95%,99%,max
node_token_length,17544.0,3.765390,7.378414,1.0,2.0,3.0,8.0,10.0,38.0,157.0
linker_token_length,17544.0,42.381498,37.747521,0.0,30.0,54.0,87.0,116.0,192.0,348.0


In [11]:
node_token_counter = Counter()

for tokens in model_df["node_tokens"]:
    node_token_counter.update(tokens)


linker_token_counter = Counter()

for tokens in model_df["linker_tokens"]:
    linker_token_counter.update(tokens)


node_vocabulary = set(node_token_counter)
linker_vocabulary = set(linker_token_counter)
combined_vocabulary = (
    node_vocabulary | linker_vocabulary
)

print("Farklı node token sayısı:", len(node_vocabulary))
print("Farklı linker token sayısı:", len(linker_vocabulary))
print("Toplam farklı token sayısı:", len(combined_vocabulary))

Farklı node token sayısı: 99
Farklı linker token sayısı: 85
Toplam farklı token sayısı: 135


In [12]:
print("En sık 20 node tokenı:")

display(
    pd.DataFrame(
        node_token_counter.most_common(20),
        columns=["token", "count"],
    )
)

print("En sık 20 linker tokenı:")

display(
    pd.DataFrame(
        linker_token_counter.most_common(20),
        columns=["token", "count"],
    )
)

En sık 20 node tokenı:


,token,count
0,[OH2],8528
1,[Zn],8326
2,[Ring1],4718
3,[Branch1],4591
4,[Cu],3587
5,<MOL_SEP>,3309
6,[C],3156
7,[OH0],2856
8,[Cd],2679
9,[OH1],2599


En sık 20 linker tokenı:


,token,count
0,[C],231334
1,[=C],100951
2,[=Branch1],84423
3,[Branch1],56853
4,[Ring1],53037
5,[=O],41578
6,[O-1],32216
7,[N],24757
8,[=N],17598
9,[O],13565


In [13]:
model_df["node_token_text"] = (
    model_df["node_tokens"]
    .apply(" ".join)
)

model_df["linker_token_text"] = (
    model_df["linker_tokens"]
    .apply(" ".join)
)

display(
    model_df[
        [
            "qmof_id",
            "node_token_text",
            "linker_token_text",
        ]
    ].head()
)

,qmof_id,node_token_text,linker_token_text
0,qmof-8a95c27,[O] <MOL_SEP> [Ba] <MOL_SEP> [Cu],[O-1] [C] [=O]
1,qmof-830ed1c,[Co],[O-1] [C] [=Branch1] [C] [=O] [C] [=C] [C] [=N...
2,qmof-5bd4a24,[Co],[O-1] [C] [=Branch1] [C] [=O] [C] [=C] [C] [=N...
3,qmof-644aab4,[Zn] [Zn],[O-1] [C] [=Branch1] [C] [=O] [C] [=C] [C] [=C...
4,qmof-eaa4957,[OH2] [Ag] [Ag] [Ag] [Ag] [OH2] <MOL_SEP> [OH2...,[O-1] [C] [=Branch1] [C] [=O] [C] [=N] [N] [=C...


In [14]:
selfies_df = model_df[
    model_df["selfies_encoding_ok"]
].copy()

print("SELFIES model veri seti:", selfies_df.shape)

SELFIES model veri seti: (17348, 24)


In [15]:
OUTPUT_PATH = (
    PROCESSED_DIR
    / "forward_model_selfies.jsonl"
)

selfies_df.to_json(
    OUTPUT_PATH,
    orient="records",
    lines=True,
    force_ascii=False,
)

print("SELFIES verisi kaydedildi:")
print(OUTPUT_PATH)

SELFIES verisi kaydedildi:
c:\Users\ASUSTUF\Desktop\tubitak_internship\data\processed\forward_model_selfies.jsonl


In [16]:
FAILURE_PATH = (
    PROCESSED_DIR
    / "selfies_encoding_failures.csv"
)

failure_df.to_csv(
    FAILURE_PATH,
    index=False,
    encoding="utf-8",
)

print("Hata raporu kaydedildi:")
print(FAILURE_PATH)

Hata raporu kaydedildi:
c:\Users\ASUSTUF\Desktop\tubitak_internship\data\processed\selfies_encoding_failures.csv


In [17]:
loaded_selfies_df = pd.read_json(
    OUTPUT_PATH,
    lines=True,
)

print("Tekrar yüklenen veri:", loaded_selfies_df.shape)
print(
    "node_tokens veri tipi:",
    type(loaded_selfies_df.loc[0, "node_tokens"]),
)

Tekrar yüklenen veri: (17348, 24)
node_tokens veri tipi: <class 'list'>


## İlk Sonuç

Node ve linker SMILES ifadeleri birbirinden ayrı tutulmuş ve
SELFIES gösterimine dönüştürülmüştür. Bir hücrede birden fazla
molekül bulunması durumunda moleküller `<MOL_SEP>` tokenı ile
ayrılmıştır.

Başarısız dönüşümler ayrı bir hata raporuna kaydedilmiş, başarılı
kayıtlar model eğitiminde kullanılmak üzere JSONL formatında
saklanmıştır. Gerçek token vocabulary'si ve padding uzunluğu,
veri sızıntısını önlemek amacıyla train-test ayrımından sonra
yalnızca training verisi üzerinden belirlenecektir.

In [18]:
print("Başarısız dönüşümlerin türleri:")

if not failure_df.empty:
    display(
        failure_df["input_type"]
        .value_counts()
        .rename("count")
        .to_frame()
    )

    print("\nEn sık hata mesajları:")

    display(
        failure_df["error"]
        .value_counts()
        .head(10)
        .rename("count")
        .to_frame()
    )
else:
    print("Başarısız dönüşüm bulunmuyor.")

Başarısız dönüşümlerin türleri:


,count
input_type,
linker,201



En sık hata mesajları:


,count
error,
kekulization failed\n\tSMILES: Cc1[nH][n-]c(c1c1c(C)[n-][nH]c1C)C,25
kekulization failed\n\tSMILES: [O-]C(=O)c1[nH][n-]c(c1)C(=O)[O-],23
kekulization failed\n\tSMILES: Cc1[nH][n-]c(c1Cc1c(C)[n-][nH]c1C)C,12
kekulization failed\n\tSMILES: [n-]1[nH]cc(c1)Cc1c[nH][n-]c1,10
kekulization failed\n\tSMILES: [O]S(c1c[nH][n-]c1)([O])[O],8
kekulization failed\n\tSMILES: Cc1cc([n-]n1CCCCn1[n-]c(cc1C)C)C,7
kekulization failed\n\tSMILES: Cc1cc([n-]n1Cc1ccc(cc1)Cn1[n-]c(cc1C)C)C,6
kekulization failed\n\tSMILES: Cc1[nH][n-]c(c1Cc1cccc(n1)Cc1c(C)[n-][nH]c1C)C,6
kekulization failed\n\tSMILES: [O-]C(=O)c1ccc(cc1)c1c(C)[n-][nH]c1C,6


In [19]:
relaxed_node_examples = model_df[
    model_df["node_relaxed_count"] > 0
][
    [
        "qmof_id",
        "smiles_nodes",
        "node_selfies_list",
    ]
].head(10)

display(relaxed_node_examples)

,qmof_id,smiles_nodes,node_selfies_list
4,qmof-eaa4957,"['[OH2][Ag][Ag][Ag][Ag][OH2]', '[OH2][Ag][Ag][...","[[OH2][Ag][Ag][Ag][Ag][OH2], [OH2][Ag][Ag][OH2]]"
6,qmof-e813edb,['[OH2][Ag][Ag]'],[[OH2][Ag][Ag]]
11,qmof-85a8986,['[OH2][Dy]([OH2])[OH2]'],[[OH2][Dy][Branch1][C][OH2][OH2]]
12,qmof-780219c,"['[OH2][La]', '[OH2][La][OH2]']","[[OH2][La], [OH2][La][OH2]]"
13,qmof-4d61b4b,"['[OH2][Pr]', '[OH2][Pr][OH2]']","[[OH2][Pr], [OH2][Pr][OH2]]"
14,qmof-d382d2b,"['[OH2][Nd]', '[OH2][Nd][OH2]']","[[OH2][Nd], [OH2][Nd][OH2]]"
15,qmof-7fb7872,"['[OH2][Sm]', '[OH2][Sm][OH2]']","[[OH2][Sm], [OH2][Sm][OH2]]"
16,qmof-be4d084,"['[OH2][Eu]', '[OH2][Eu][OH2]']","[[OH2][Eu], [OH2][Eu][OH2]]"
17,qmof-2b207a3,"['[OH2][Gd]', '[OH2][Gd][OH2]']","[[OH2][Gd], [OH2][Gd][OH2]]"
18,qmof-96fb808,['[OH2][La][OH2]'],[[OH2][La][OH2]]


In [20]:
relaxed_linker_examples = model_df[
    model_df["linker_relaxed_count"] > 0
][
    [
        "qmof_id",
        "smiles_linkers",
        "linker_selfies_list",
    ]
].head(10)

display(relaxed_linker_examples)

,qmof_id,smiles_linkers,linker_selfies_list
47,qmof-9b03351,['[O-]C(=O)C[NH](CC(=O)O)CCC[NH](CC(=O)O)CC(=O...,[[O-1][C][=Branch1][C][=O][C][NH1][Branch1][#B...
104,qmof-df4cc2a,['[O-]C(=O)C1CC(C[NH2]1)O'],[[O-1][C][=Branch1][C][=O][C][C][C][Branch1][B...
111,qmof-dcabab2,['[O-]C(=O)c1cccc(c1)N(=O)=O'],[[O-1][C][=Branch1][C][=O][C][=C][C][=C][C][=B...
146,qmof-b9a1228,['O=N(=O)c1ccc(cc1)OP(=O)(Oc1ccc(cc1)N(=O)=O)[...,[[O][=N][=Branch1][C][=O][C][=C][C][=C][Branch...
196,qmof-741f030,['O[C](O[Ca]1([OH2])([OH2])[O]C(=O)c2cccc(C(=O...,[[O][CH0][Branch2][Ring1][#C][O][Ca][Branch1][...
201,qmof-45689c4,['ClCCn1nnnc1[N]N(=O)=O'],[[Cl][C][C][N][N][=N][N][=C][Ring1][Branch1][N...
202,qmof-0a8808d,['[N]=[N]=NCCn1nnnc1[N]N(=O)=O'],[[NH0][=NH0][=N][C][C][N][N][=N][N][=C][Ring1]...
216,qmof-83e5a32,"['NC1=[N]=C(N=N1)N', '[O-]C(=O)C=CC(=O)[O-]']",[[N][C][=NH0][=C][Branch1][Branch1][N][=N][Rin...
244,qmof-ebae1e4,['[Cd][N]12CN3CN(C2)CN(C1)C3'],[[Cd][NH0][C][N][C][N][Branch1][Ring2][C][Ring...
335,qmof-34b5f2d,"['C(Cc1ccncc1)Cc1ccncc1', '[O-]C(=O)c1cc(SSc2c...",[[C][Branch1][#Branch2][C][C][=C][C][=N][C][=C...


### SELFIES Dönüşüm Hataları

Toplam 17.544 kaydın 17.348'i başarıyla SELFIES gösterimine
dönüştürülmüş, 196 kayıt dönüştürülememiştir. Başarı oranı
%98,88'dir.

Başarısız dönüşümlerin büyük bölümü, yüklü azot atomları içeren
aromatik halkalarda görülen `kekulization failed` hatalarından
kaynaklanmaktadır. Bu SMILES ifadeleri kimyasal yapıyı değiştirme
riski nedeniyle otomatik veya elle düzeltilmemiştir.

Başarısız kayıtlar ana veriden silinmemiş, ayrı bir hata raporunda
saklanmıştır. SELFIES tabanlı ilk model deneylerinde yalnızca
başarıyla dönüştürülen 17.348 kayıt kullanılacaktır.